In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors


In [2]:
df= pd.read_csv("homework_7.1.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  10000 non-null  int64  
 1   X           10000 non-null  float64
 2   W           10000 non-null  float64
 3   Z           10000 non-null  float64
 4   Y           10000 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 390.8 KB


question 1

In [4]:
import statsmodels.api as sm

# Select variables
X = df[['X']]
W = df['W']
Z = df['Z']
Y = df['Y']

# Compute the error term if the true model is:
# Y = X + Z + W + error
error = Y - X['X'] - W - Z

# Correlation between X and the error term
corr = X['X'].corr(error)

print("Correlation between X and the error term:", corr)

Correlation between X and the error term: 0.33568570286549565


q.2 

In [5]:
import pandas as pd

# If your dataset is called data and has columns X, W, Z, Y

# Model written as: Y = X + Z + error
# So error = Y - X - Z
df["error_XZ_model"] = df["Y"] - df["X"] - df["Z"]

# Correlation of X with the error term
corr = df["X"].corr(df["error_XZ_model"])

print("Correlation between X and error term:", corr)

Correlation between X and error term: 0.546341401917415


q3.

In [7]:
import statsmodels.api as sm

# Width of the window around each W value
tolerance = 0.2

for w_value in [-1, 0, 1]:

    # Keep observations with W close to the desired value
    subset = df[(df["W"] >= w_value - tolerance) &
                  (df["W"] <= w_value + tolerance)]

    # Regress Y on X and Z
    X_reg = sm.add_constant(subset[["X", "Z"]])
    y_reg = subset["Y"]

    model = sm.OLS(y_reg, X_reg).fit()

    print(f"\nW ≈ {w_value}")
    print(f"Number of observations: {len(subset)}")
    print(f"Coefficient of X: {model.params['X']:.4f}")


W ≈ -1
Number of observations: 967
Coefficient of X: 0.8699

W ≈ 0
Number of observations: 1546
Coefficient of X: 1.4177

W ≈ 1
Number of observations: 909
Coefficient of X: 1.9343


q4.

In [8]:

def make_error(corr_const, num):
    err = []
    prev = np.random.normal(0, 1)
    for n in range(num):
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1)
        err.append(prev)
    return np.array(err)

n = 1000
num_trials = 200

for corr_const in [0.2, 0.5, 0.8]:

    beta_estimates = []
    se_estimates = []

    for _ in range(num_trials):

        # Covariate
        W = np.random.normal(0, 1, n)

        # Correlated errors
        err_x = make_error(corr_const, n)
        err_y = make_error(corr_const, n)

        # Treatment
        X = W + err_x

        # Outcome
        Y = 1 + 2*X + 3*W + err_y

        # Regression with intercept
        X_reg = sm.add_constant(np.column_stack((X, W)))

        model = sm.OLS(Y, X_reg).fit()

        beta_estimates.append(model.params[1])   # coefficient of X
        se_estimates.append(model.bse[1])         # standard error of X

    print(f"\nCorrelation constant = {corr_const}")
    print("Std. dev. of beta estimates:",
          np.std(beta_estimates))
    print("Mean estimated standard error:",
          np.mean(se_estimates))


Correlation constant = 0.2
Std. dev. of beta estimates: 0.03202508833814273
Mean estimated standard error: 0.031715093248546496

Correlation constant = 0.5
Std. dev. of beta estimates: 0.0393981026352081
Mean estimated standard error: 0.031629050350284356

Correlation constant = 0.8
Std. dev. of beta estimates: 0.07008789892215767
Mean estimated standard error: 0.03168923037863465
